In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import string
import joblib

In [4]:
fake=pd.read_csv('fake.csv')
true=pd.read_csv('true.csv')

In [5]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [6]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [7]:
fake['class']=0
true['class']=1

In [9]:
data=pd.concat([fake,true],axis=0)

In [10]:
data.sample(10)

,title,text,subject,date,class
365,"Billy Bush: It was Trump's voice on ""Access Ho...",WASHINGTON (Reuters) - Former NBC television h...,politicsNews,"December 4, 2017",1
1220,Comey Might Have Just Made Sure We Don’t End ...,Many of us have been ready to impeach Donald T...,News,"June 8, 2017",0
700,Factbox: How the U.S. Senate and House tax pro...,(Reuters) - A tax plan by U.S. Senate Republic...,politicsNews,"November 9, 2017",1
16792,Mugabe removed as WHO goodwill envoy after out...,GENEVA/LONDON (Reuters) - Zimbabwean President...,worldnews,"October 22, 2017",1
4030,Pentagon probes Trump's ex-adviser Flynn over ...,WASHINGTON (Reuters) - The Pentagon inspector ...,politicsNews,"April 27, 2017",1
2485,Angry Caller LOSES IT When Asked To Prove Med...,People who parrot and spread Trump s claims th...,News,"February 17, 2017",0
12383,Czech PM Babis takes office but so far lacks p...,PRAGUE (Reuters) - Czech Prime Minister Andrej...,worldnews,"December 13, 2017",1
20973,[VIDEO] THEY BURNED DOWN CITIES…14 Yr Old Dous...,Barack and Michelle Obama had some unusual ho...,left-news,"Feb 20, 2016",0
730,"U.S. firm Air Products, China's Yankuang plan ...","BEIJING (Reuters) - A top Chinese coal miner, ...",politicsNews,"November 9, 2017",1
444,WATCH: Kellyanne Conway’s Latest Gushing Abou...,The wicked witch of the White House makes Amer...,News,"September 3, 2017",0


In [11]:
data=data.drop(["title","subject","date"],axis=1)

In [12]:
data.reset_index(inplace=True)

In [13]:
data.drop(['index'],axis=1,inplace=True)

In [14]:
data.sample(5)

,text,class
7009,Republicans repeatedly tell us our schools wou...,0
4570,"Every four years, we re told that the upcoming...",0
25393,WASHINGTON (Reuters) - When the U.S. Congress ...,1
19988,Does anyone else find it ironic that black mul...,0
1378,Last night Buzzfeed reported that a massive do...,0


In [18]:
def clean_text(text):
    text=text.lower()
    text=re.sub('\[.*?\]',"",text)
    text=re.sub("\\W"," ",text)
    text=re.sub("https?:://\S|www\.\S+","",text)
    text=re.sub("<.*?>+","",text)
    text=re.sub("[%s]"%re.escape(string.punctuation),"",text)
    text=re.sub("\n","",text)
    text=re.sub("\w*\d\w","",text)
    return text


In [19]:
data["text"]=data["text"].apply(clean_text)

In [21]:
x=data["text"]
y=data["class"]

xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.25,random_state=42)

In [29]:
vectorizer=TfidfVectorizer()
xv_train=vectorizer.fit_transform(xtrain)
xv_test=vectorizer.transform(xtest)


In [30]:
lr=LogisticRegression()
lr.fit(xv_train,ytrain)


LogisticRegression()

In [31]:
predicttion=lr.predict(xv_test)
lr.score(xv_test,ytest)

0.985478841870824

In [32]:
print(classification_report(ytest,predicttion))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      5895
           1       0.98      0.99      0.98      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [33]:
joblib.dump(vectorizer,"vectorizer.jb")
joblib.dump(lr,"lr_model.jb")

['lr_model.jb']